# Chapter 12: Putting it on a real phone line

This notebook runs every piece of code in the chapter, from the saved results, in order. Read the chapter alongside it: [Putting it on a real phone line](https://books.sugiv.fyi/voice_applications/real-phone-line.html).


In [ ]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the notebook
# from a clone of it. Either way it moves into the book's folder and installs what the chapter needs.
import os
import pathlib
import sys

if "google.colab" in sys.modules and not pathlib.Path("/content/voices-of-the-future").exists():
    !git clone -q --depth 1 https://github.com/sugix/voices-of-the-future /content/voices-of-the-future
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/voices-of-the-future")]
root = next(p for p in candidates if (p / "voice" / "sim").is_dir())
os.chdir(root / "voice")
%pip install -q -r ../requirements/results.txt

## How a phone call reaches your code

In [ ]:
import base64
import json
import sys

import numpy as np
from scipy import signal

sys.path.insert(0, "sim")
from wav import load, mulaw_decode  # Chapter 2's reader and decoder


def mulaw_encode(x):
    """G.711 mu-law, the inverse of Chapter 2's mulaw_decode: float samples in, one byte per sample out."""
    s = np.clip(np.round(x * 32768), -32635, 32635).astype(np.int32)
    sign = np.where(s < 0, 0x80, 0)
    m = np.abs(s) + 0x84
    exponent = np.floor(np.log2(m)).astype(np.int32) - 7
    mantissa = (m >> (exponent + 3)) & 0x0F
    return (~(sign | (exponent << 4) | mantissa) & 0xFF).astype(np.uint8)


reply, sr = load("data/ch6/listen/kokoro_r05_spoken.wav")  # a receptionist reply from Chapter 6
phone = signal.resample_poly(reply, 8000, sr)
wire = mulaw_encode(phone)
back = mulaw_decode(wire.tobytes())
snr = 10 * np.log10(np.sum(phone ** 2) / np.sum((phone - back) ** 2))
frames = [wire[i:i + 160] for i in range(0, len(wire), 160)]  # 160 bytes = 20 ms at 8 kHz
print(f"{len(phone) / 8000:.1f} s of reply = {len(wire):,} bytes = {len(frames)} packets of 20 ms; "
      f"round trip through mu-law: {snr:.1f} dB SNR")
message = {"event": "media", "streamSid": "MZ...", "media": {"payload": base64.b64encode(frames[0]).decode()}}
print(json.dumps(message)[:150], "...")

## The receptionist as a stream

## How many calls can one GPU carry?

In [ ]:
import matplotlib.pyplot as plt

load_llm = json.load(open("data/ch12/llm_load.json"))
rows = []
for lv in load_llm["levels"]:
    r = lv["requests"]
    fs = np.array([x["first_sentence"] for x in r if x["first_sentence"]])
    ft = np.array([x["first_token"] for x in r if x["first_token"]])
    rows.append((lv["in_flight"], np.median(ft), np.median(fs), np.percentile(fs, 90), len(r) / lv["wall"]))
print("in flight  first token  first sentence (median, 90th pct)  requests/s")
for n, ft, fs, fs90, rps in rows:
    print(f"{n:9d} {ft:11.3f} s {fs:12.2f} s {fs90:7.2f} s {rps:16.1f}")

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.plot([r[0] for r in rows], [r[2] for r in rows], "o-", label="median")
ax.plot([r[0] for r in rows], [r[3] for r in rows], "o--", label="90th percentile")
ax.set(xscale="log", xticks=[r[0] for r in rows], xticklabels=[str(r[0]) for r in rows],
       xlabel="requests in flight at once", ylabel="first sentence (s)",
       title="Qwen3-4B on vLLM, one 20 GB GPU")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
talk = json.load(open("data/ch8/talk_4b-snap-guarded.json"))["calls"]
hear = json.load(open("data/ch8/hear.json"))
model_s, turn_s = 0.0, 0.0
for turns in talk.values():
    for t in turns:
        if not t["steps"]:
            continue
        caller = hear[t["clip"]]["speech_end"] - 0.3
        thinking = sum(s["timing"]["total"] for s in t["steps"])
        speaking = len(t["reply"].split()) / 2.6  # Chapter 6: about 2.6 words a second from the voice
        model_s += thinking
        turn_s += caller + (hear[t["clip"]]["endpoint_wait"] or 1.2) + thinking + speaking
duty = model_s / turn_s
print(f"the language model is busy for {duty:.0%} of a call")
for n, ft, fs, fs90, rps in rows:
    if n in (16, 32, 64):
        print(f"{n} requests in flight at once is roughly {n / duty:.0f} simultaneous calls")

In [ ]:
asr = json.load(open("data/ch12/asr_load.json"))
print("Parakeet-TDT, Chapter 8's 65 caller turns:")
for a in asr:
    print(f"   batch {a['batch']:3d}: {1000 * a['wall'] / a['clips']:4.0f} ms of GPU per turn, "
          f"{a['audio_s'] / a['wall']:5.0f}x real time")
tts = json.load(open("data/ch12/tts_load.json"))
print("\nKokoro on the GPU, first audio of a reply's first sentence:")
for k in tts["kokoro_gpu"]:
    first = [x for x, _ in k["requests"]]
    print(f"   {k['parallel']} at once: median {np.median(first):.2f} s, 90th percentile {np.percentile(first, 90):.2f} s")
print("\nPocket TTS on CPU processes (2 threads each), first audio:")
for k in tts["pocket_cpu"]:
    print(f"   {k['processes']} at once: median {np.median([x[0] for x in k['requests']]):.2f} s, "
          f"{np.median([x[2] / x[1] for x in k['requests']]):.1f}x real time each")